# Control flow, functions, and scope

By this point you already know how to write an `if`, a loop, and a function. The intermediate step is understanding the deeper rules behind them: how names are resolved across scopes, how closures capture variables, how function signatures shape APIs, and how control-flow choices affect readability.

Scope is especially important because many bugs are really name-resolution mistakes. A function can read from an outer scope, shadow a name from outside, or capture a changing value in a closure. Without a clear model of that behaviour, code may look reasonable while doing something surprising.

As you read each notebook, try to explain which scope each name belongs to and why Python resolves it there. That is the foundation for writing reliable functions later.

## Visual model

```text
global scope
  -> function scope
       -> inner function scope
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Loops, and the `else` nobody expects

In [ ]:
for item in collection:      # iterates ANYTHING iterable (Module 14)
    ...

for i, item in enumerate(collection, start=1):
    ...

for a, b in zip(xs, ys, strict=True):    # strict=True is 3.10+ and you want it
    ...

for key, value in mapping.items():
    ...

`zip(strict=True)` raises if the iterables have different lengths. Without it,
`zip` silently stops at the shortest, which has hidden many data bugs. Default
to `strict=True` unless truncation is genuinely intended.

### `for ... else`

The `else` clause runs **if the loop completed without `break`**. It is not "if
the loop body never ran".

In [ ]:
for user in users:
    if user.is_admin:
        print("found an admin")
        break
else:
    print("no admin found")        # runs only if we never broke out

Read `else` here as `nobreak` and it becomes obvious. It exists to remove the
`found = False` flag variable:

In [ ]:
found = False                       # the pattern for ... else replaces
for user in users:
    if user.is_admin:
        found = True
        break
if not found:
    ...

It is rare in real code, and it is on every Python quiz.

### Loop control

```text
break        # exit the innermost loop
continue     # next iteration
```


There is no labelled break. To exit nested loops, either extract the loops into
a function and `return`, or use a flag, or iterate a product:

In [ ]:
from itertools import product
for i, j in product(range(n), range(m)):
    if done(i, j):
        break                       # one loop, so one break is enough

Extracting to a function is almost always the cleanest of the three.

### Do not mutate what you are iterating

In [ ]:
for x in items:
    if pred(x):
        items.remove(x)             # silently skips elements (Module 02, q11)

items = [x for x in items if not pred(x)]      # correct
items[:] = [x for x in items if not pred(x)]   # correct, and in place

---

## Concept 3. `match`: structural pattern matching, not a switch

`match` (3.10+) destructures values. Using it as a C-style switch wastes it.

In [ ]:
match command.split():
    case ["go", direction]:
        move(direction)
    case ["take", *items]:                 # capture the rest
        for item in items:
            take(item)
    case ["quit" | "exit"]:                # alternatives
        raise SystemExit
    case []:
        print("say something")
    case _:                                 # the default; _ matches anything
        print(f"unknown: {command}")

It matches structure, types, and attributes:

In [ ]:
match event:
    case {"type": "click", "pos": (x, y)}:          # dict + tuple shape
        handle_click(x, y)
    case {"type": "key", "code": int() as code}:    # type check + capture
        handle_key(code)
    case Point(x=0, y=0):                            # class patterns
        print("origin")
    case Point(x=x, y=y) if x == y:                  # a guard
        print("diagonal")

Two traps:

**A bare name is a capture, not a comparison.**

```text
case OK:              # binds anything to the name OK. Always matches!
case Status.OK:       # a dotted name IS compared. This is what you meant.
```


This is the number one `match` bug. Any pattern that is a plain identifier
captures; only dotted names, literals, and class patterns compare.

**Class patterns need `__match_args__`** for positional matching, which
`@dataclass` provides automatically (Module 11).

When is `match` worth it? When you are destructuring nested data — parsing,
protocol handling, AST walking, event dispatch. For dispatching on a single
value, a dict of functions is clearer and faster.

---

## Concept 4. Functions: the six kinds of parameter

In [ ]:
def f(pos_only, /, standard, *args, kw_only, **kwargs):
    ...

| Kind | Declared | Called as |
|---|---|---|
| Positional-only | before `/` | `f(1)` only |
| Positional-or-keyword | between `/` and `*` | `f(1)` or `f(standard=1)` |
| Var-positional | `*args` | extra positionals collected into a tuple |
| Keyword-only | after `*` | `f(kw_only=1)` only |
| Var-keyword | `**kwargs` | extra keywords collected into a dict |

In [ ]:
def connect(host, port=5432, /, *, timeout=30, retries=3, **options):
    ...

connect("db", 5432, timeout=10, ssl=True)      # ok
connect(host="db")                              # TypeError: host is positional-only
connect("db", 5432, 10)                         # TypeError: timeout is keyword-only

**Why bother?**

- `/` (positional-only) frees you to rename parameters later without breaking
  callers. The standard library uses it heavily for exactly this reason.
- `*` (keyword-only) forces call sites to be readable. `resize(img, 800, 600,
  True, False)` is unreadable; `resize(img, width=800, height=600,
  preserve_aspect=True, upscale=False)` is not.

**Rule of thumb: any boolean parameter should be keyword-only.** A bare `True`
at a call site carries no information.

### Arguments are unpacked, not copied

In [ ]:
args = (1, 2)
kwargs = {"c": 3}
f(*args, **kwargs)          # equivalent to f(1, 2, c=3)

### The mutable default, again

```text
def f(items=[]):            # WRONG -- evaluated once at def time (Module 02)
def f(items=None):          # right
    if items is None:
        items = []
```


Same for `{}`, `set()`, `datetime.now()`, and any expression whose value should
be per-call. `ruff` rule `B006` catches it.

### Functions are objects

In [ ]:
def greet(name): return f"hi {name}"

greet.__name__          # 'greet'
greet.__doc__           # the docstring
greet.__defaults__      # the default values tuple
greet.__annotations__   # the type hints, as a dict

handlers = {"greet": greet}          # store them
def apply(fn, x): return fn(x)       # pass them
def make(): return greet             # return them

This is what makes decorators, callbacks, and higher-order functions possible,
and it is the subject of Module 15.

---

## Concept 7. Type hints

Hints are not enforced at runtime. They are checked by mypy or pyright, read by
your editor, and used by libraries like Pydantic and FastAPI. Module 17 is the
full treatment; this is the working subset.

In [ ]:
def greet(name: str, times: int = 1) -> str: ...

def parse(raw: str) -> dict[str, int]: ...            # builtin generics, 3.9+
def find(xs: list[int]) -> int | None: ...            # union syntax, 3.10+
def apply(fn: Callable[[int], str], x: int) -> str: ...

from collections.abc import Iterable, Sequence
def total(values: Iterable[float]) -> float: ...      # accept ANY iterable

Two habits worth forming now:

**Accept the widest type, return the narrowest.** Take `Iterable[str]`, not
`list[str]` — then a generator, a tuple, or a set all work. Return `list[str]`,
not `Iterable[str]` — then the caller knows they can index it.

**`X | None` is not optional-as-in-omittable**; it means the value may be `None`.
A parameter is omittable because it has a default.

Write hints on every function you write in this course. Not because Python needs
them, but because writing the return type forces you to decide what the function
actually produces — which is where half of all design bugs are found.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Statements and expressions
- Section 2: Loops, and the `else` nobody expects
- Section 3: `match`: structural pattern matching, not a switch
- Section 4: Functions: the six kinds of parameter
- Section 5: Scope: LEGB
- Section 6: Closures
- Section 7: Type hints

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Any

---

## `Status`

_Status_

In [ ]:
class Status(Enum):
    OK = "ok"
    ERROR = "error"
    PENDING = "pending"

---

## `Point`

_Point_

In [ ]:
@dataclass
class Point:
    x: int
    y: int

---

## `run_command`

Parse a command line with `match` on line.split().

In [ ]:
def run_command(line: str) -> str:
    """Parse a command line with `match` on line.split().

      "go north"            -> "moving north"
      "take sword shield"   -> "taking: sword, shield"
      "take"                -> "take what?"
      "look"                -> "you see nothing"
      "quit" or "exit"      -> "goodbye"
      "" (empty)            -> "say something"
      anything else         -> "I do not understand 'xyz'"

    Use sequence patterns, a star pattern, and an alternation. Do NOT use
    if/elif -- the point is to practise the patterns.
    """
    raise NotImplementedError

---

## `describe_event`

Match on the SHAPE of a dict.

In [ ]:
def describe_event(event: dict[str, Any]) -> str:
    """Match on the SHAPE of a dict.

      {"type": "click", "pos": (0, 0)}          -> "click at origin"
      {"type": "click", "pos": (x, y)}          -> "click at 3,4"
      {"type": "key", "code": 27}               -> "escape"
      {"type": "key", "code": <int>}            -> "key 65"
      {"type": "key", "code": <non-int>}        -> "bad key event"
      {"type": "scroll", "delta": <negative>}   -> "scroll up 3"   (use a guard)
      {"type": "scroll", "delta": <positive>}   -> "scroll down 3"
      anything else                             -> "unknown event"

    Note that a mapping pattern matches on a SUBSET of keys -- extra keys in the
    event do not prevent a match. Verify that, then say in a comment whether
    that is the behaviour you want for an event router and why.
    """
    raise NotImplementedError

---

## `classify_point`

Class patterns.

In [ ]:
def classify_point(p: Point) -> str:
    """Class patterns.

      Point(0, 0)    -> "origin"
      Point(x, 0)    -> "on the x axis"
      Point(0, y)    -> "on the y axis"
      Point(x, x)    -> "on the diagonal"      (use a guard: x == y)
      otherwise      -> "at 3,4"

    @dataclass generates __match_args__, which is what makes POSITIONAL class
    patterns like Point(0, 0) work. Try removing @dataclass and see the error;
    that tells you what __match_args__ is for.
    """
    raise NotImplementedError

---

## `check_status_BROKEN`

This function is WRONG. Run it with Status.PENDING, see what happens,

In [ ]:
def check_status_BROKEN(status: Status) -> str:
    """This function is WRONG. Run it with Status.PENDING, see what happens,
    then explain why in a comment before fixing it below.

    This is the single most common `match` bug.

    Bonus experiment: move `case OK:` ABOVE `case Status.ERROR:` and try to run
    the file. CPython refuses to compile it, with a message that tells you
    exactly what is wrong. Why can the compiler catch that arrangement and not
    this one?
    """
    OK = Status.OK  # noqa: N806
    match status:
        case Status.ERROR:
            return "failed"
        case OK:
            return "all good"

---

## `check_status`

TODO: the corrected version.

In [ ]:
def check_status(status: Status) -> str:
    """TODO: the corrected version."""
    raise NotImplementedError

---

## `test_run_command`

_test run command_

In [ ]:
def test_run_command() -> None:
    assert run_command("go north") == "moving north"
    assert run_command("take sword shield") == "taking: sword, shield"
    assert run_command("take") == "take what?"
    assert run_command("look") == "you see nothing"
    assert run_command("quit") == "goodbye"
    assert run_command("exit") == "goodbye"
    assert run_command("") == "say something"
    assert run_command("dance wildly") == "I do not understand 'dance wildly'"

---

## `test_describe_event`

_test describe event_

In [ ]:
def test_describe_event() -> None:
    assert describe_event({"type": "click", "pos": (0, 0)}) == "click at origin"
    assert describe_event({"type": "click", "pos": (3, 4)}) == "click at 3,4"
    assert describe_event({"type": "key", "code": 27}) == "escape"
    assert describe_event({"type": "key", "code": 65}) == "key 65"
    assert describe_event({"type": "key", "code": "a"}) == "bad key event"
    assert describe_event({"type": "scroll", "delta": -3}) == "scroll up 3"
    assert describe_event({"type": "scroll", "delta": 3}) == "scroll down 3"
    assert describe_event({"type": "mystery"}) == "unknown event"
    # extra keys must not prevent a match
    assert describe_event(
        {"type": "click", "pos": (1, 2), "timestamp": 999}) == "click at 1,2"

---

## `test_classify_point`

_test classify point_

In [ ]:
def test_classify_point() -> None:
    assert classify_point(Point(0, 0)) == "origin"
    assert classify_point(Point(5, 0)) == "on the x axis"
    assert classify_point(Point(0, 5)) == "on the y axis"
    assert classify_point(Point(3, 3)) == "on the diagonal"
    assert classify_point(Point(3, 4)) == "at 3,4"

---

## `test_status`

_test status_

In [ ]:
def test_status() -> None:
    assert check_status_BROKEN(Status.ERROR) == "failed"
    assert check_status_BROKEN(Status.PENDING) == "all good", (
        "if this fails the trap has been fixed in the BROKEN version; "
        "restore it so you can see the bug"
    )
    assert check_status(Status.OK) == "all good"
    assert check_status(Status.ERROR) == "failed"
    assert check_status(Status.PENDING) == "pending"

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    for t in tests:
        t()
        print(f"  PASS  {t.__name__}")
    print(f"\n{len(tests)} tests passed")

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.